# Camada Gold no Databricks

Este notebook cria o modelo dimensional em Star Schema para a camada Gold, lendo diretamente as tabelas da camada Silver no Unity Catalog e gravando as tabelas finais em Delta.

## Objetivo
- Ler as tabelas da camada Silver
- Construir as dimensões de estudante, professor, turma e disciplina
- Preparar a tabela de fato de notas
- Gravar tudo no schema `gold` do catálogo Databricks


In [ ]:
# ==============================================================================
# 1. LEITURA DA CAMADA SILVER
# ==============================================================================

# Leitura direta das tabelas já persistidas em Unity Catalog
# A estrutura esperada no catálogo é: silver.<tabela>
df_students = spark.read.table("silver.students")
df_teacher = spark.read.table("silver.teacher")
df_classes = spark.read.table("silver.classes")
df_subjects = spark.read.table("silver.subjects")
df_grades = spark.read.table("silver.grades")

print("Tabelas da camada Silver lidas com sucesso.")

In [ ]:
from pyspark.sql import functions as F


def drop_audit_columns(df):
    """Remove colunas de auditoria que não pertencem ao modelo final."""
    cols_to_remove = [c for c in df.columns if c.upper() in {"DATA_HORA_SILVER", "NOME_TABELA"}]
    return df.drop(*cols_to_remove) if cols_to_remove else df


def first_matching_column(df, candidates):
    """Retorna a primeira coluna encontrada em qualquer variação de casing."""
    upper_map = {c.upper(): c for c in df.columns}
    for candidate in candidates:
        if candidate.upper() in upper_map:
            return upper_map[candidate.upper()]
    return None


def build_dimension(df, id_candidates, name_candidates, id_alias, name_alias):
    """Cria uma dimensão com identificador e nome, preservando o nome final desejado."""
    id_col = first_matching_column(df, id_candidates)
    name_col = first_matching_column(df, name_candidates)

    if id_col is None:
        raise ValueError(f"Nenhuma coluna de identificador encontrada. Candidatos: {id_candidates}")

    selected = [F.col(id_col).cast("string").alias(id_alias)]

    if name_col is not None:
        selected.append(F.col(name_col).cast("string").alias(name_alias))
    else:
        selected.append(F.lit(None).cast("string").alias(name_alias))

    return df.select(*selected).dropDuplicates()


df_students = drop_audit_columns(df_students)
df_teacher = drop_audit_columns(df_teacher)
df_classes = drop_audit_columns(df_classes)
df_subjects = drop_audit_columns(df_subjects)
df_grades = drop_audit_columns(df_grades)

print("Colunas de auditoria removidas com sucesso.")

In [ ]:
# ==============================================================================
# 2. CONSTRUÇÃO DAS DIMENSÕES
# ==============================================================================

# O schema da camada Silver usa nomes como IDENTIFIER, STUDENT_IDENTIFIER, TEACHER_IDENTIFIER...
dim_student = build_dimension(
    df_students,
    ["IDENTIFIER", "STUDENT_IDENTIFIER", "ID_STUDENT", "STUDENT_ID", "IDENTIFIER_STUDENT"],
    ["NAME", "NOME", "NOME_COMPLETO", "NOME_ESTUDANTE"],
    "student_id",
    "student_name"
)

dim_teacher = build_dimension(
    df_teacher,
    ["IDENTIFIER", "TEACHER_IDENTIFIER", "ID_TEACHER", "TEACHER_ID", "IDENTIFIER_TEACHER"],
    ["NAME", "NOME", "NOME_PROFESSOR", "NOME_COMPLETO"],
    "teacher_id",
    "teacher_name"
)

dim_class = build_dimension(
    df_classes,
    ["IDENTIFIER", "CLASS_IDENTIFIER", "ID_CLASS", "CLASS_ID", "IDENTIFIER_CLASS"],
    ["NAME", "NOME", "NOME_TURMA", "DESCRICAO"],
    "class_id",
    "class_name"
)

dim_subject = build_dimension(
    df_subjects,
    ["IDENTIFIER", "SUBJECT_IDENTIFIER", "ID_SUBJECT", "SUBJECT_ID", "IDENTIFIER_SUBJECT"],
    ["NAME", "NOME", "NOME_DISCIPLINA", "DESCRICAO"],
    "subject_id",
    "subject_name"
)

display(dim_student.limit(5))
display(dim_teacher.limit(5))
display(dim_class.limit(5))
display(dim_subject.limit(5))

print("Dimensões montadas com sucesso.")

In [ ]:
# ==============================================================================
# 3. CONSTRUÇÃO DA TABELA DE FATO
# ==============================================================================

# A camada Silver converteu os IDs para nomes do tipo STUDENT_IDENTIFIER, TEACHER_IDENTIFIER...
student_fk = first_matching_column(df_grades, ["STUDENT_IDENTIFIER", "IDENTIFIER", "ID_STUDENT", "STUDENT_ID", "IDENTIFIER_STUDENT"])
teacher_fk = first_matching_column(df_grades, ["TEACHER_IDENTIFIER", "IDENTIFIER", "ID_TEACHER", "TEACHER_ID", "IDENTIFIER_TEACHER"])
class_fk = first_matching_column(df_grades, ["CLASS_IDENTIFIER", "IDENTIFIER", "ID_CLASS", "CLASS_ID", "IDENTIFIER_CLASS"])
subject_fk = first_matching_column(df_grades, ["SUBJECT_IDENTIFIER", "IDENTIFIER", "ID_SUBJECT", "SUBJECT_ID", "IDENTIFIER_SUBJECT"])
grade_value = first_matching_column(df_grades, ["GRADE", "NOTA", "GRADE_VALUE", "NOTA_FINAL", "VALOR_NOTA"])

missing_columns = [
    name for name, value in {
        "student_fk": student_fk,
        "teacher_fk": teacher_fk,
        "class_fk": class_fk,
        "subject_fk": subject_fk,
        "grade_value": grade_value,
    }.items() if value is None
]

if missing_columns:
    raise ValueError(f"Colunas essenciais da tabela grades não foram encontradas: {missing_columns}")

fact_grades = (
    df_grades
    .select(
        F.col(student_fk).cast("string").alias("student_id"),
        F.col(teacher_fk).cast("string").alias("teacher_id"),
        F.col(class_fk).cast("string").alias("class_id"),
        F.col(subject_fk).cast("string").alias("subject_id"),
        F.col(grade_value).cast("double").alias("grade_value")
    )
    .dropDuplicates()
)

display(fact_grades.limit(10))

print("Tabela fato montada com sucesso.")

In [ ]:
# ==============================================================================
# 4. GRAVAÇÃO NA CAMADA GOLD
# ==============================================================================

# Grava em formato Delta no catálogo Unity Catalog
# O schema gold foi criado na célula inicial
# Ajuste o catálogo para o correto se necessário

dim_student.write.format("delta").mode("overwrite").saveAsTable("gold.dim_student")
dim_teacher.write.format("delta").mode("overwrite").saveAsTable("gold.dim_teacher")
dim_class.write.format("delta").mode("overwrite").saveAsTable("gold.dim_class")
dim_subject.write.format("delta").mode("overwrite").saveAsTable("gold.dim_subject")
fact_grades.write.format("delta").mode("overwrite").saveAsTable("gold.fact_grades")

print("Tabelas da camada Gold gravadas com sucesso no catálogo do Databricks!")

display(spark.sql("SHOW TABLES IN gold"))


## Observação final

Este notebook foi ajustado para seguir o padrão correto de execução no Databricks:
- leitura via `spark.read.table()`
- escrita via `write.format("delta").saveAsTable()`
- ausência de caminhos locais ou bibliotecas de filesystem
- uso do Unity Catalog e do schema `gold`

Se o ambiente tiver um catálogo diferente de `main`, ajuste a linha `USE CATALOG main` para o catálogo correto.